In [1]:
pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import pandas as pd

In [3]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def tokenise_with_alignment(text, tokenizer):
    words = text.strip().split()

    subword_ids = []
    tokens = []
    word_first_subword = []

    current_index = 0

    for word in words:
        pieces = tokenizer.tokenize(word)

        if not pieces:
            pieces = [tokenizer.unk_token]

        word_first_subword.append(current_index)

        tokens.extend(pieces)
        subword_ids.extend(tokenizer.convert_tokens_to_ids(pieces))

        current_index += len(pieces)

    return words, tokens, subword_ids, word_first_subword

In [6]:
train_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/train.parquet")
val_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/val.parquet")
test_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/test.parquet")

In [7]:
train_df

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,"Dear HR, please initiate the process to update...","Dear HR, please initiate the process to update...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-USER...","[dear, hr, ,, please, initiate, the, process, ..."
1,Notre [ACCOUNTNAME_1] doit se conformer aux ré...,Notre Savings Account doit se conformer aux ré...,"[O, B-ACCOUNTNAME, I-ACCOUNTNAME, O, O, O, O, ...","[notre, savings, account, doi, ##t, se, confor..."
2,"""Our office, located at [STREETADDRESS_1], [CI...","""Our office, located at 70729 Hildegard Pine, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","["", our, office, ,, located, at, 70, ##7, ##29..."
3,Nous aimerions organiser une réunion avec un/u...,Nous aimerions organiser une réunion avec un/u...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[no, ##us, aim, ##eri, ##ons, organise, ##r, u..."
4,To ensure we remain connected throughout the o...,To ensure we remain connected throughout the o...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[to, ensure, we, remain, connected, throughout..."
...,...,...,...,...
17325,"[FIRSTNAME_1], could you guide [USERNAME_1] on...","Delphine, could you guide Lucio.Fritsch on how...","[O, O, O, O, O, O, B-USERNAME, I-USERNAME, I-U...","[del, ##phine, ,, could, you, guide, luc, ##io..."
17326,We need your signature on the document sent pr...,We need your signature on the document sent pr...,"[O, O, O, O, O, O, O, O, O, O, B-EMAIL, I-EMAI...","[we, need, your, signature, on, the, document,..."
17327,Pourriez-vous vous assurer que le navigateur w...,Pourriez-vous vous assurer que le navigateur w...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[pour, ##rie, ##z, -, vo, ##us, vo, ##us, assu..."
17328,"Pour maintenir nos normes éthiques, nous avons...","Pour maintenir nos normes éthiques, nous avons...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[pour, main, ##ten, ##ir, nos, norm, ##es, et,..."


In [8]:
def prepare_dataset(df):
    all_sentences = []
    
    for i in range(len(df)):
        words, tokens, _, word_first_subword = tokenise_with_alignment(
            df["unmasked_text"].iloc[i],
            tokenizer
        )
    
        subword_labels = df["token_entity_labels"].iloc[i]
    
        sentence_words = []
        sentence_labels = []
    
        for w, idx in zip(words, word_first_subword):
            if idx < len(subword_labels):
                label = subword_labels[idx]
            else:
                label = "O"
    
            sentence_words.append(w)
            sentence_labels.append(label)
    
        # keep only valid sentences
        if len(sentence_words) == len(sentence_labels):
            all_sentences.append([sentence_words, sentence_labels])

    return pd.DataFrame(all_sentences, columns=["words", "labels"])

In [9]:
train_df=prepare_dataset(train_df)

In [10]:
val_df=prepare_dataset(val_df)

In [11]:
test_df=prepare_dataset(test_df)

In [12]:
train_sentences = []
s=set()
for _, row in train_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    for val in token_entity_labels:
        s.add(val)
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    train_sentences.append(sentence)

In [13]:
val_sentences = []
for _, row in val_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    for val in token_entity_labels:
        s.add(val)
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    val_sentences.append(sentence)

In [14]:
test_sentences = []
for _, row in test_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    for val in token_entity_labels:
        s.add(val)
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    test_sentences.append(sentence)

In [15]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  
        
    
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [16]:
def extract_features(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [17]:
X_train1, y_train1 = extract_features(train_sentences)
X_val1, y_val1 = extract_features(val_sentences)
X_test1, y_test1 = extract_features(test_sentences)

In [18]:
crf1 = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf1.fit(X_train1, y_train1)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

In [19]:
y_pred = crf1.predict(X_val1)

print(metrics.flat_classification_report(
    y_val1, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.978     1.000     0.989        87
   B-ACCOUNTNUMBER      0.685     0.485     0.568       103
B-CREDITCARDNUMBER      0.623     0.447     0.521        85
           B-EMAIL      0.947     0.984     0.966       128
            B-IPV4      0.588     0.472     0.524       106
            B-IPV6      0.684     0.299     0.416        87
             B-MAC      0.935     0.430     0.589       100
        B-PASSWORD      0.891     0.527     0.662        93
    B-PHONE_NUMBER      0.947     0.807     0.871        88
             B-SSN      0.841     0.569     0.679        65
        B-USERNAME      0.954     0.466     0.626       133
     I-ACCOUNTNAME      0.974     0.974     0.974       153
    I-PHONE_NUMBER      0.946     0.957     0.951        92
             I-SSN      0.972     0.921     0.946        38
                 O      0.993     0.998     0.996     57818

          accuracy                    

In [20]:
from joblib import dump, load
dump(crf1, "crf_model1.joblib")

['crf_model1.joblib']

# additional features

In [21]:
def word2features2(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "word.has_dot()": "." in word,
        "word.has_digit()": any(c.isdigit() for c in word),
        "word.has_colon()": ":" in word,
        "word.is_hex()": bool(re.fullmatch(r"[0-9a-fA-F]+", word)),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  

    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [22]:
def extract_features2(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features2(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [23]:
X_train2, y_train2 = extract_features2(train_sentences)
X_val2, y_val2 = extract_features2(val_sentences)
X_test2, y_test2 = extract_features2(test_sentences)

In [24]:
crf2 = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=200,
    all_possible_transitions=True
)

crf2.fit(X_train2, y_train2)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=200)

In [25]:
y_pred = crf2.predict(X_val2)

print(metrics.flat_classification_report(
    y_val2, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.978     1.000     0.989        87
   B-ACCOUNTNUMBER      0.658     0.505     0.571       103
B-CREDITCARDNUMBER      0.629     0.459     0.531        85
           B-EMAIL      1.000     0.992     0.996       128
            B-IPV4      0.694     0.642     0.667       106
            B-IPV6      0.733     0.632     0.679        87
             B-MAC      0.771     0.740     0.755       100
        B-PASSWORD      0.831     0.581     0.684        93
    B-PHONE_NUMBER      0.926     0.852     0.888        88
             B-SSN      0.837     0.554     0.667        65
        B-USERNAME      0.974     0.556     0.708       133
     I-ACCOUNTNAME      0.974     0.974     0.974       153
    I-PHONE_NUMBER      0.948     0.989     0.968        92
             I-SSN      0.950     1.000     0.974        38
                 O      0.995     0.998     0.997     57818

          accuracy                    

In [26]:
from joblib import dump, load
dump(crf2, "crf_model2.joblib")

['crf_model2.joblib']

# additional features

In [27]:
def word_shape(word):
    shape = []
    for c in word:
        if c.isupper():
            shape.append("X")
        elif c.islower():
            shape.append("x")
        elif c.isdigit():
            shape.append("d")
        else:
            shape.append(c)
    return "".join(shape)

In [28]:
def word2features3(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,

        # lexical
        "word.lower()": word.lower(),
        "word[:2]": word[:2],
        "word[:3]": word[:3],
        "word[-2:]": word[-2:],
        "word[-3:]": word[-3:],

        # casing
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.islower()": word.islower(),

        # length
        "word.len": len(word),

        # shape
        "word.shape": word_shape(word),

        # character composition
        "word.has_digit()": any(c.isdigit() for c in word),
        "word.has_lower()": any(c.islower() for c in word),
        "word.has_upper()": any(c.isupper() for c in word),
        "word.has_special()": any(not c.isalnum() for c in word),

        "digit_count": sum(c.isdigit() for c in word),
        "alpha_count": sum(c.isalpha() for c in word),
        "special_count": sum(not c.isalnum() for c in word),

        # separators
        "word.has_dot()": "." in word,
        "word.has_colon()": ":" in word,
        "word.has_dash()": "-" in word,
        "word.has_slash()": "/" in word,
        "word.has_at()": "@" in word,

        # regex patterns
        "looks_email": bool(
            re.fullmatch(r".+@.+\..+", word)
        ),

        "looks_ipv4": bool(
            re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", word)
        ),

        "looks_mac": bool(
            re.fullmatch(
                r"[0-9A-Fa-f]{2}([-:][0-9A-Fa-f]{2}){5}",
                word
            )
        ),

        "looks_hex": bool(
            re.fullmatch(r"[0-9a-fA-F]{8,}", word)
        ),

        "four_digits": bool(
            re.fullmatch(r"\d{4}", word)
        ),

        "word.is_hex()": bool(
            re.fullmatch(r"[0-9a-fA-F]+", word)
        ),
    }

    # previous token
    if i > 0:
        prev = sent[i - 1][0]

        features.update({
            "-1:word.lower()": prev.lower(),
            "-1:word.shape": word_shape(prev),
            "-1:word.len": len(prev),
            "-1:word.istitle()": prev.istitle(),
            "-1:word.isupper()": prev.isupper(),
            "-1:has_digit": any(c.isdigit() for c in prev),

            "-1:+0": prev.lower() + "_" + word.lower(),
        })

    else:
        features["BOS"] = True

    # next token
    if i < len(sent) - 1:
        nxt = sent[i + 1][0]

        features.update({
            "+1:word.lower()": nxt.lower(),
            "+1:word.shape": word_shape(nxt),
            "+1:word.len": len(nxt),
            "+1:word.istitle()": nxt.istitle(),
            "+1:word.isupper()": nxt.isupper(),
            "+1:has_digit": any(c.isdigit() for c in nxt),

            "+0:+1": word.lower() + "_" + nxt.lower(),
        })

    else:
        features["EOS"] = True

    return features

In [29]:
def extract_features3(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features3(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [30]:
X_train3, y_train3 = extract_features3(train_sentences)
X_val3, y_val3 = extract_features3(val_sentences)
X_test3, y_test3 = extract_features3(test_sentences)

In [31]:
crf3 = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=150,
    all_possible_transitions=True
)

crf3.fit(X_train3, y_train3)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=150)

In [32]:
y_pred3 = crf3.predict(X_val3)

print(metrics.flat_classification_report(
    y_val3, y_pred3, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.978     1.000     0.989        87
   B-ACCOUNTNUMBER      1.000     1.000     1.000       103
B-CREDITCARDNUMBER      0.738     0.729     0.734        85
           B-EMAIL      0.992     1.000     0.996       128
            B-IPV4      0.758     0.858     0.805       106
            B-IPV6      0.778     0.805     0.791        87
             B-MAC      1.000     0.980     0.990       100
        B-PASSWORD      1.000     0.946     0.972        93
    B-PHONE_NUMBER      0.989     1.000     0.994        88
             B-SSN      0.985     1.000     0.992        65
        B-USERNAME      0.991     0.842     0.911       133
     I-ACCOUNTNAME      0.968     0.974     0.971       153
    I-PHONE_NUMBER      1.000     1.000     1.000        92
             I-SSN      1.000     1.000     1.000        38
                 O      0.999     0.999     0.999     57818

          accuracy                    

# test

In [33]:
y_pred = crf3.predict(X_test3)

print(metrics.flat_classification_report(
    y_test3, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.990     1.000     0.995       102
   B-ACCOUNTNUMBER      1.000     0.991     0.995       110
B-CREDITCARDNUMBER      0.705     0.753     0.728        89
           B-EMAIL      0.987     1.000     0.994       153
            B-IPV4      0.719     0.833     0.772       120
            B-IPV6      0.732     0.781     0.756       105
             B-MAC      0.987     0.987     0.987        77
        B-PASSWORD      0.979     0.931     0.954       101
    B-PHONE_NUMBER      1.000     0.991     0.995       106
             B-SSN      0.978     0.978     0.978        89
        B-USERNAME      0.969     0.855     0.908       145
     I-ACCOUNTNAME      0.978     0.989     0.983       179
    I-PHONE_NUMBER      1.000     1.000     1.000       108
             I-SSN      1.000     0.958     0.979        48
                 O      0.998     0.998     0.998     63885

          accuracy                    

In [34]:
from joblib import dump, load
dump(crf3, "crf_model3.joblib")

['crf_model3.joblib']